# Darcy with user-written mixed H(div) equations

RT0 flux, P0 pressure and a local boundary pressure form the local coefficient
vector $(q,p,\eta)$. Signed P0 skeletal coefficients prescribe physical normal
Darcy flux. The augmented local equations are

$$
\begin{aligned}
Mq-D^Tp+N\eta &= 0,\\
-Dq &= -f,\\
N^Tq-F\lambda &= 0.
\end{aligned}
$$

The global pressure-test row is $C=-B^T$ and uses the explicit Dirichlet boundary
functional. The constant pressure/boundary-pressure kernel is retained; its
physical pressure integral selects the local complement. For
$p=1+x^2+y^2$, $q=-2(x,y)$ and $f=-4$, RT0 represents flux exactly while
the P0 pressure has its stated projection error.


In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "src/pymhm").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
from scipy import sparse
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble, columns, rows
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import SkeletonSpace
from pymhm.fem.scalar.operators import rt0_operators, face_integration, boundary_data
from examples.tutorial_local_provider import exact_pressure, source, TutorialResult, diagnostics

mesh = TriangleMesh.unit_square()
skeleton = SkeletonSpace(mesh)

def local_equations(cell):
    """Write the flux, pressure and boundary-flux equations without a PDE solver."""
    fine = mesh.submesh(cell, 2)
    mass, divergence, force = rt0_operators(fine, 1.0, source, order=4)
    _, flux_map = face_integration(mesh, cell, fine, skeleton)
    nq, npres, nb = len(fine.faces), len(fine.cells), len(fine.boundary_faces)
    normal = sparse.coo_matrix((np.ones(nb), (fine.boundary_faces, np.arange(nb))),
                               shape=(nq, nb)).tocsc()
    zero = sparse.csc_matrix((npres, nb))
    a = sparse.bmat([[mass, -divergence.T, normal],
                    [-divergence, None, zero], [normal.T, zero.T, None]], format="csc")
    b = np.zeros((nq + npres + nb, flux_map.shape[1]))
    b[nq + npres:] = -flux_map
    L = np.r_[np.zeros(nq), -force, np.zeros(nb)]
    kernel = np.r_[np.zeros(nq), np.ones(npres + nb)][:, None]
    physical_mean = np.r_[np.zeros(nq), fine.areas, np.zeros(nb)][:, None]
    return LocalEquations(
        a, L, columns(*b.T), rows(*(-b.T)), skeleton.cell_dofs(cell),
        kernel=kernel, moments=physical_mean / (kernel.T @ physical_mean).item(),
        metadata=(fine, physical_mean[:, 0]),
    )

boundary, _ = boundary_data(skeleton, exact_pressure, order=4)
problem = MultiscaleProblem(
    Equation(0, np.r_[boundary, np.zeros(len(mesh.cells))]), local_equations,
    range(len(mesh.cells)), skeleton.size, (1,) * len(mesh.cells),
)
system = assemble(problem)
solution = system.solve()
record = diagnostics(TutorialResult(problem, system, solution, "mixed", "dirichlet", "basix"))
print(record)
assert record["field_L2_errors"]["Darcy_flux"] < 1e-10
assert record["field_L2_errors"]["divergence"] < 1e-10


## Compare the available local spaces

Select names below to run a smaller comparison. All records identify the
executed pressure, flux and normal trace spaces. `classical-rt1` has a globally
conforming flux and no MHM skeleton. The H(div) patches demonstrate the
implemented families; they do not establish a uniform inf-sup bound or reproduce
the heterogeneous well campaigns.

The comparison below selects declared configurations from the application catalogue. Its drivers assemble the public mathematical providers in [examples/formulations](../../examples/formulations/) through the same generic local/global API. Each configuration records its approximation spaces and boundary conventions.


In [ ]:
from examples.tutorial_scalar_variants import run_variant

selected_methods = ('rt0', 'rt1', 'classical-rt1', 'bdm', 'bdm-plus', 'bdm-double-plus', 'tensor-rt', 'tensor-rt-plus', 'hdiv-tetra', 'hdiv-tetra-plus', 'hdiv-prism')
records = [run_variant(name) for name in selected_methods]
for record in records:
    print(record["variant"], record["spaces_and_conventions"], record["metrics"])
rt0 = next(record for record in records if record["variant"] == "rt0")
assert np.isclose(rt0["metrics"]["scalar_l2"], np.sqrt(7 / 18), rtol=0, atol=3e-14)
